# Phase 3 / p3_03 -- R4a/R4b: noise-augmented training

Trains `banglabert` and `char_ngram` (the two models the augmentation ablation
is about -- METHODOLOGY M5.2 / Figure_Plan F5) on BOTH augmentation groups
(`augmented_n1n4` = group A, `augmented_n5n8` = group B), then runs the same
inline R2/R6 noise grid as p3_01 so `pipeline/p3_figures.fig_r4b_augmentation`
has matched AND mismatched cells for both training directions.

36 jobs total: 2 models x 3 tasks x 3 seeds x 2 augmentation variants.

Same resumability contract as p3_01/p3_02. Run p3_02 with `JOB_FAMILY="r4"`
afterwards if a session is cut off mid-grid.

In [ ]:
# torch is intentionally NOT reinstalled -- Kaggle's GPU image ships a
# CUDA-matched build; see requirements-phase3.txt's top comment.
%pip install -q "transformers>=4.40" "scikit-learn>=1.4" "statsmodels>=0.14" \
    "regex==2024.11.6" \
    "git+https://github.com/csebuetnlp/normalizer@d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9"

import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible -- on Kaggle, enable a GPU accelerator in "
         "Notebook Settings before running a real (non --limit) job.")


In [ ]:
import os, sys, shutil, time

# ---------------------------------------------------------------------------
# Kaggle bootstrap. Before running this notebook on Kaggle, attach THREE
# private Datasets via the "Add Data" panel (see PHASE3_STATUS.md for exact
# creation steps):
#   1. bangla-noisebench-data-final -- data/final/ (SentNoB is CC-BY-ND-4.0,
#      this Dataset MUST be private)
#   2. bangla-noisebench-results    -- the versioned results.csv/curves.csv/
#      preds/ state, so a killed session can resume for free
#   3. bangla-noisebench-code       -- this repository, so `pipeline` and
#      `noisebench` are importable (Kaggle notebooks don't see the repo
#      checkout directly)
# Change the three slugs below if yours differ.
# ---------------------------------------------------------------------------
DATA_DATASET_SLUG = "bangla-noisebench-data-final"
RESULTS_DATASET_SLUG = "bangla-noisebench-results"
CODE_DATASET_SLUG = "bangla-noisebench-code"

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    CODE_ROOT = f"/kaggle/input/{CODE_DATASET_SLUG}/bangla-noisebench"
    sys.path.insert(0, CODE_ROOT)
    DATA_FINAL_ROOT = f"/kaggle/input/{DATA_DATASET_SLUG}/final"
    RESULTS_SRC = f"/kaggle/input/{RESULTS_DATASET_SLUG}"
    RESULTS_DIR = "/kaggle/working/results"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    for fname in ("results.csv", "curves.csv"):
        src = os.path.join(RESULTS_SRC, fname)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(RESULTS_DIR, fname))
            print(f"[bootstrap] restored {fname} from {RESULTS_SRC}")
        else:
            print(f"[bootstrap] no prior {fname} in {RESULTS_SRC} -- starting fresh")
    preds_src = os.path.join(RESULTS_SRC, "preds")
    if os.path.isdir(preds_src):
        shutil.copytree(preds_src, os.path.join(RESULTS_DIR, "preds"), dirs_exist_ok=True)
        print(f"[bootstrap] restored preds/ ({len(os.listdir(os.path.join(RESULTS_DIR, 'preds')))} files)")
else:
    _ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    sys.path.insert(0, _ROOT)
    DATA_FINAL_ROOT = os.path.join(_ROOT, "data", "final")
    RESULTS_DIR = os.path.join(_ROOT, "results")
    print(f"[bootstrap] not on Kaggle -- using local repo paths under {_ROOT}")

print(f"DATA_FINAL_ROOT = {DATA_FINAL_ROOT}")
print(f"RESULTS_DIR     = {RESULTS_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------
JOB_SLICE = None  # e.g. "0:18", "18:36"


In [ ]:
# ---------------------------------------------------------------------------
# Print how many jobs remain and the estimated wall time BEFORE starting --
# required by the Phase 3 spec so a session is never launched blind.
# ---------------------------------------------------------------------------
from pipeline import p3_train, p3_estimate

ALL_JOBS = p3_train.make_r4_jobs()
if JOB_SLICE:
    start, stop = (int(x) if x else None for x in JOB_SLICE.split(":"))
    ALL_JOBS = ALL_JOBS[start:stop]
    print(f"JOB_SLICE={JOB_SLICE!r} applied -- {len(ALL_JOBS)} of the full R4 list")

results_path = os.path.join(RESULTS_DIR, "results.csv")
existing = p3_train.load_existing_result_keys(results_path)
remaining = []
for job in ALL_JOBS:
    clean_done = p3_train._results_key(job.run_id, "on", None, 0) in existing
    grid_done = p3_train._job_grid_is_complete(job, existing)
    if not (clean_done and grid_done):
        remaining.append(job)

print(f"{len(ALL_JOBS)} R4 jobs total, {len(ALL_JOBS) - len(remaining)} already "
     "fully done in results.csv, "
     f"{len(remaining)} remaining this session.")

est = [p3_estimate.estimate_job(j, DATA_FINAL_ROOT) for j in remaining]
total_gpu_h = sum(e.gpu_hours for e in est)
total_wall_h = sum(e.total_seconds for e in est) / 3600.0
print(f"Estimated: {total_gpu_h:.1f} GPU-h, {total_wall_h:.1f} wall-clock h for the "
     "remaining jobs (p3_estimate's throughput ASSUMPTIONS -- calibrate against "
     "logged epoch elapsed= times on the first real run, see p3_estimate.py docstring).")
if total_wall_h > 10.5:
    print("WARNING: estimated wall time exceeds the 11h session budget "
         "(SESSION_CAP_HOURS) -- this notebook run will likely be cut off "
         "mid-grid. That's fine (resumable), but consider a smaller JOB_SLICE.")


In [ ]:
# ---------------------------------------------------------------------------
# Train -> evaluate -> append -> delete checkpoint -> next. Never persists a
# checkpoint past its own job (constraint 0); results.csv is written
# incrementally so a session killed mid-grid loses at most the in-flight cell.
# ---------------------------------------------------------------------------
progress = p3_train.Progress("notebook")
for i, job in enumerate(remaining, 1):
    print(f"\n=== job {i}/{len(remaining)}: {job.run_id} ===")
    t0 = time.time()
    p3_train.run_job(job, DATA_FINAL_ROOT, RESULTS_DIR, progress, run_noise_grid=True)
    print(f"job wall time: {time.time() - t0:.1f}s")


In [ ]:
# ---------------------------------------------------------------------------
# Re-upload results.csv/curves.csv/preds/ as a NEW VERSION of the private
# results Dataset, so state survives this session's death (12h cap). Requires
# a kaggle.json API token already configured in this environment (Kaggle
# notebooks have one by default when "Internet" is on); the `kaggle` CLI is
# preinstalled on Kaggle images.
# ---------------------------------------------------------------------------
import subprocess, json as _json

if IS_KAGGLE:
    upload_dir = "/kaggle/working/results_upload"
    os.makedirs(upload_dir, exist_ok=True)
    for fname in ("results.csv", "curves.csv"):
        src = os.path.join(RESULTS_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(upload_dir, fname))
    preds_dir = os.path.join(RESULTS_DIR, "preds")
    if os.path.isdir(preds_dir):
        shutil.copytree(preds_dir, os.path.join(upload_dir, "preds"), dirs_exist_ok=True)

    meta_path = os.path.join(upload_dir, "dataset-metadata.json")
    if not os.path.exists(meta_path):
        with open(meta_path, "w", encoding="utf-8") as fh:
            _json.dump({"title": RESULTS_DATASET_SLUG, "id": f"YOUR_KAGGLE_USERNAME/{RESULTS_DATASET_SLUG}",
                       "licenses": [{"name": "unknown"}]}, fh)
        print(f"WROTE {meta_path} with a PLACEHOLDER id -- edit YOUR_KAGGLE_USERNAME "
             "before the first version push, or `kaggle datasets init` this dir once by hand.")

    try:
        out = subprocess.run(
            ["kaggle", "datasets", "version", "-p", upload_dir,
            "-m", f"session update {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}",
            "-r", "zip", "--dir-mode", "zip"],
            capture_output=True, text=True, timeout=600)
        print(out.stdout)
        if out.returncode != 0:
            print("kaggle datasets version FAILED -- stderr:", out.stderr)
            print("Fix: make sure this Dataset already exists once (create it manually "
                 "the first time via the Kaggle UI, private, then this cell versions it "
                 "on every subsequent run) and that kaggle.json is present.")
    except FileNotFoundError:
        print("`kaggle` CLI not found -- re-upload manually: download "
             f"{upload_dir} and push it as a new private Dataset version.")
else:
    print("[upload] not on Kaggle -- skipping dataset re-upload (results stayed in", RESULTS_DIR, ")")
